In [ ]:
# ============================================================================
# LIBRARIES
# ============================================================================
import os
import sys
import time
import json
import warnings
from pathlib import Path
from typing import Tuple, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, callbacks, optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import AdamW

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, max_error, make_scorer
from sklearn.exceptions import DataConversionWarning
from scikeras.wrappers import KerasRegressor

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DataConversionWarning)

# Set plot style
sns.set_style("ticks")
plt.rcParams['font.family'] = 'sans-serif'

# ============================================================================
# TERMINAL COLOR CODES
# ============================================================================
class Colors:
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    WARNING = '\033[93m'  # Same as YELLOW
    SUCCESS = '\033[92m'  # Same as GREEN
    
    @classmethod
    def header(cls, text): 
        print(f"\n{cls.HEADER}{cls.BOLD}{'='*80}\n{text.center(80)}\n{'='*80}{cls.ENDC}\n")
    
    @classmethod
    def success(cls, text): print(f"{cls.GREEN}✓ {text}{cls.ENDC}")
    
    @classmethod
    def warning(cls, text): print(f"{cls.YELLOW}⚠ {text}{cls.ENDC}")
    
    @classmethod
    def error(cls, text): print(f"{cls.RED}✗ {text}{cls.ENDC}")
    
    @classmethod
    def info(cls, text): print(f"{cls.CYAN}ℹ {text}{cls.ENDC}")
    
    @classmethod
    def step(cls, num, text): 
        print(f"\n{cls.BLUE}{cls.BOLD}[STEP {num}] {text}\n{'-'*80}{cls.ENDC}")

# ============================================================================
# FILE MANAGEMENT
# ============================================================================
class FileManager:
    """Handle all file operations with user-friendly interface"""
    
    @staticmethod
    def get_data_files(folder_path: str) -> List[Tuple[int, str, str]]:
        """Get all supported data files in folder"""
        supported_ext = ['.csv', '.xlsx', '.xls', '.txt']
        files = []
    
        for file in Path(folder_path).iterdir():
            if file.suffix.lower() in supported_ext:
                files.append((file.name, str(file)))
        files = sorted(files, key=lambda x: x[0])
        return [(idx, name, path) for idx, (name, path) in enumerate(files, 1)]
    
    @staticmethod
    def select_data_file() -> Tuple[str, Optional[str]]:
        """Interactive file and sheet selection"""
        Colors.step(1, "SELECT DATA FILE")
        
        # Get folder path
        while True:
            folder = input(f"{Colors.CYAN}Enter folder path containing data files: {Colors.ENDC}").strip().strip('"\'')
            if os.path.isdir(folder): break
            Colors.error("Invalid folder path. Please try again.")
        
        # List data files
        data_files = FileManager.get_data_files(folder)
        
        if not data_files:
            Colors.error("No supported data files found!")
            sys.exit(1)
        
        print(f"\n{Colors.BOLD}Available data files:{Colors.ENDC}")
        for num, name, _ in data_files:
            print(f"  {Colors.GREEN}[{num}]{Colors.ENDC} {name}")
        
        # Select file
        while True:
            try:
                sel = int(input(f"\n{Colors.CYAN}Select file number: {Colors.ENDC}"))
                if 1 <= sel <= len(data_files):
                    file_path = data_files[sel-1][2]
                    file_name = data_files[sel-1][1]
                    Colors.success(f"Selected: {file_name}")
                    break
                Colors.error(f"Enter number between 1 and {len(data_files)}")
            except ValueError:
                Colors.error("Please enter a valid number")
        
        # Handle Excel sheets
        sheet_name = None
        if file_path.lower().endswith(('.xlsx', '.xls')):
            try:
                xls = pd.ExcelFile(file_path)
                if len(xls.sheet_names) > 1:
                    print(f"\n{Colors.BOLD}Available sheets:{Colors.ENDC}")
                    for i, sheet in enumerate(xls.sheet_names, 1):
                        print(f"  {Colors.GREEN}[{i}]{Colors.ENDC} {sheet}")
                    
                    while True:
                        try:
                            sel = int(input(f"\n{Colors.CYAN}Select sheet number: {Colors.ENDC}"))
                            if 1 <= sel <= len(xls.sheet_names):
                                sheet_name = xls.sheet_names[sel-1]
                                Colors.success(f"Selected sheet: {sheet_name}")
                                break
                        except ValueError: pass
                else:
                    sheet_name = xls.sheet_names[0]
                    Colors.info(f"Using sheet: {sheet_name}")
            except Exception as e:
                Colors.error(f"Error reading Excel file: {e}")
                sys.exit(1)
        
        return file_path, sheet_name
    
    @staticmethod
    def setup_output_directory() -> Tuple[str, str]:
        Colors.step("OUTPUT", "CONFIGURE SAVE LOCATION")
        choice = input(f"{Colors.CYAN}Save in home directory? (y/n): {Colors.ENDC}").strip().lower()
        if choice in ['y', 'yes']:
            base_dir = os.path.join(os.path.expanduser('~'), 'DNN')
        else:
            while True:
                user_dir = input(f"{Colors.CYAN}Enter custom directory path: {Colors.ENDC}").strip().strip('"\'')
                if os.path.exists(user_dir) and os.path.isdir(user_dir):
                    base_dir = os.path.join(user_dir, 'DNN')
                    break
                Colors.error("Invalid directory path")
        os.makedirs(base_dir, exist_ok=True)
        # Get save name
        while True:
            save_name = input(f"{Colors.CYAN}Enter results name (no extension): {Colors.ENDC}").strip()
            if not save_name:
                Colors.error("Name cannot be empty")
            elif any(c in save_name for c in r'\/:*?"<>|'):
                Colors.error("Invalid characters in name")
            else:
                full_path = os.path.join(base_dir, save_name + "_Results.xlsx")
                if os.path.exists(full_path):
                    overwrite = input(f"{Colors.YELLOW}File exists. Overwrite? (y/n): {Colors.ENDC}").strip().lower()
                    if overwrite == 'y':
                        break
                else:
                    break
        Colors.success(f"Output location: {base_dir}")
        return base_dir, save_name

# ============================================================================
# DATA LOADER
# ============================================================================
class DataLoader:
    """Load and preprocess data"""
    
    @staticmethod
    def load_file(file_path: str, sheet_name: Optional[str] = None) -> pd.DataFrame:
        """Load data file"""
        ext = file_path.lower().split('.')[-1]
        
        try:
            if ext == 'csv':
                df = pd.read_csv(file_path)
            elif ext in ['xls', 'xlsx']:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
            elif ext == 'txt':
                df = pd.read_csv(file_path, delimiter='\t', engine='python')
            else:
                Colors.error(f"Unsupported format: .{ext}")
                return None
        except Exception as e:
            Colors.error(f"Failed to load file: {e}")
            return None
        
        if df.isnull().values.any():
            Colors.error("Dataset contains missing values!")
            sys.exit(1)
        
        Colors.success(f"File loaded successfully ({len(df)} rows)")
        return df
    
    @staticmethod
    def load_symbol_conversion(location: str, col_name: str) -> Optional[str]:
        """Parse symbol_conversion.txt and return LaTeX symbol for a column name"""
        # Determine file path
        if os.path.isdir(location):
            file_path = os.path.join(location, 'symbol_conversion.txt')
        else:
            file_path = location
    
        if not os.path.isfile(file_path):
            Colors.error(f"symbol_conversion.txt not found at: {file_path}")
            return None
    
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                if '=' in line:
                    key, _, value = line.partition('=')
                    key = key.strip()
                    value = value.strip()
                    if key == col_name:
                        return value
    
        Colors.warning(f"No symbol found for '{col_name}', using column name as label")
        return col_name

    @staticmethod
    def select_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, str]:
        """Interactive column selection"""
        Colors.step(2, "SELECT INPUT/OUTPUT COLUMNS")
        
        print(f"\n{Colors.BOLD}Available columns:{Colors.ENDC}")
        for i, col in enumerate(df.columns, 1):
            print(f"  [{i}] {col}")
        
        # Option to drop columns
        drop = input(f"\n{Colors.CYAN}Drop any columns before selection? (y/n): {Colors.ENDC}").strip().lower()
        
        if drop in ['y', 'yes']:
            while True:
                print(f"\n{Colors.BOLD}Columns: {Colors.ENDC}{', '.join(df.columns)}")
                cols_drop = input(f"{Colors.CYAN}Enter column names to drop (comma-separated): {Colors.ENDC}").strip().split(',')
                cols_drop = [c.strip() for c in cols_drop]
                
                invalid = [c for c in cols_drop if c not in df.columns]
                if invalid:
                    Colors.error(f"Invalid columns: {', '.join(invalid)}")
                else:
                    df = df.drop(columns=cols_drop)
                    Colors.success(f"Dropped: {', '.join(cols_drop)}")
                    break
            
            print(f"\n{Colors.BOLD}Updated columns:{Colors.ENDC}")
            for i, col in enumerate(df.columns, 1):
                print(f"  [{i}] {col}")
        
        # Select input columns
        while True:
            print(f"\n{Colors.BOLD}Columns: {Colors.ENDC}{', '.join(df.columns)}")
            input_cols = input(f"\n{Colors.CYAN}Input (X) columns (comma-separated): {Colors.ENDC}").strip().split(',')
            input_cols = [c.strip() for c in input_cols]
            
            if all(c in df.columns for c in input_cols):
                break
            Colors.error("Invalid column names. Try again.")
        
        # Select output column
        while True:
            print(f"\n{Colors.BOLD}Columns: {Colors.ENDC}{', '.join(df.columns)}")
            output_col = input(f"{Colors.CYAN}Output (y) column: {Colors.ENDC}").strip()
            if output_col not in df.columns or output_col in input_cols:
                Colors.error("Invalid or duplicate output column")
                continue
    
            # Ask about symbol_conversion.txt
            use_sym = input(f"{Colors.CYAN}Use symbol_conversion.txt for LaTeX label? (y/n): {Colors.ENDC}").strip().lower()
            if use_sym in ['y', 'yes']:
                while True:
                    location = input(f"{Colors.CYAN}Enter folder or full path to symbol_conversion.txt: {Colors.ENDC}").strip().strip('"\'')
                    output_latex = DataLoader.load_symbol_conversion(location, output_col)
                    if output_latex:
                        Colors.success(f"LaTeX label: {output_latex}")
                        break
                    retry = input(f"{Colors.CYAN}Try a different path? (y/n): {Colors.ENDC}").strip().lower()
                    if retry not in ['y', 'yes']:
                        Colors.info(f"Using column name '{output_col}' as label")
                        output_latex = output_col
                        break
            else:
                Colors.info(f"Using column name '{output_col}' as label")
                output_latex = output_col
            break
        
        X = df[input_cols]
        y = df[output_col]
        
        Colors.success(f"Input features: {len(input_cols)}")
        Colors.success(f"Output: {output_col}")
        
        return X, y, output_latex

# ============================================================================
# DATA PROCESSOR
# ============================================================================
class DataProcessor:
    """Handle data scaling and splitting"""
    
    def __init__(self, test_size: float = 0.3, random_state: int = 42):
        self.test_size = test_size
        self.random_state = random_state
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
    
    def scale_fit(self, X: pd.DataFrame, y: pd.Series) -> Tuple[np.ndarray, np.ndarray]:
        """Fit scalers and transform data"""
        X_scaled = self.scaler_X.fit_transform(X)
        y_scaled = self.scaler_y.fit_transform(y.values.reshape(-1, 1))
        return X_scaled, y_scaled
    
    def split_data(self, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Split data into train/test"""
        return train_test_split(X, y, test_size=self.test_size, random_state=self.random_state)
    
    @staticmethod
    def determine_test_size(n_samples: int) -> float:
        """Automatically determine appropriate test size"""
        if n_samples < 20:
            return 0.15
        elif n_samples < 30:
            return 0.20
        elif n_samples < 40:
            return 0.25
        else:
            return 0.30

# ============================================================================
# MODEL BUILDER
# ============================================================================
class DNNBuilder:
    """Build DNN models with configurable architecture"""
    
    @staticmethod
    def build_model(input_dim: int, hidden_layers: tuple, learning_rate: float = 0.001) -> keras.Model:
        """Build DNN model"""
        model = models.Sequential()
        
        # First hidden layer
        model.add(layers.Dense(
            hidden_layers[0],
            input_shape=(input_dim,),
            kernel_initializer='glorot_uniform',
            kernel_regularizer=regularizers.l2(0.0001)
        ))
        model.add(layers.PReLU())
        
        # Additional hidden layers
        for neurons in hidden_layers[1:]:
            model.add(layers.Dense(
                neurons,
                kernel_initializer='glorot_uniform',
                kernel_regularizer=regularizers.l2(0.0001)
            ))
            model.add(layers.PReLU())
        
        # Output layer
        model.add(layers.Dense(1))
        
        # Compile
        optimizer = optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-5)
        model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])
        
        return model
    
    @staticmethod
    def calculate_parameters(model: keras.Model) -> Tuple[int, Dict]:
        """Calculate trainable parameters"""
        total_params = 0
        breakdown = {}
        
        for i, layer in enumerate(model.layers):
            if hasattr(layer, 'trainable_weights') and len(layer.trainable_weights) > 0:
                layer_params = sum([tf.size(w).numpy() for w in layer.trainable_weights])
                total_params += layer_params
                
                if isinstance(layer, tf.keras.layers.Dense):
                    weights_shape = layer.trainable_weights[0].shape
                    weights_count = int(weights_shape[0] * weights_shape[1])
                    biases_count = int(layer.trainable_weights[1].shape[0]) if len(layer.trainable_weights) > 1 else 0
                    
                    breakdown[f'Layer_{i}_{layer.name}'] = {
                        'weights': weights_count,
                        'biases': biases_count,
                        'total': layer_params
                    }
        
        return total_params, breakdown

# ============================================================================
# METRICS CALCULATOR
# ============================================================================
class MetricsCalculator:
    """Calculate performance metrics"""
    
    @staticmethod
    def calculate_all(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
        """Calculate all metrics"""
        return {
            'R2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': root_mean_squared_error(y_true, y_pred),
            'MaxError': max_error(y_true, y_pred)
        }

# ============================================================================
# HYPERPARAMETER MANAGER
# ============================================================================
class HyperparameterManager:
    """Manage hyperparameter selection"""
    
    @staticmethod
    def get_manual_params() -> Dict:
        """Get manual hyperparameters from user"""
        Colors.step(3, "CONFIGURE HYPERPARAMETERS (MANUAL MODE)")
        
        print(f"{Colors.BOLD}Note:{Colors.ENDC} All values are case-sensitive")
        print(f"{Colors.WARNING}Changing defaults may affect results{Colors.ENDC}\n")
        
        def get_int(prompt, min_val, max_val, default):
            while True:
                try:
                    val = int(input(f"{Colors.CYAN}{prompt} (Range: {min_val}-{max_val}, Suggested: {default}): {Colors.ENDC}"))
                    if min_val <= val <= max_val:
                        return val
                    Colors.error(f"Enter value between {min_val} and {max_val}")
                except ValueError:
                    Colors.error("Enter a valid integer")
        
        def get_float(prompt, min_val, max_val, default):
            while True:
                try:
                    val = float(input(f"{Colors.CYAN}{prompt} (Range: {min_val}-{max_val}, Suggested: {default}): {Colors.ENDC}"))
                    if min_val <= val <= max_val:
                        return val
                    Colors.error(f"Enter value between {min_val} and {max_val}")
                except ValueError:
                    Colors.error("Enter a valid number")
        
        def get_hidden_layers():
            layers = []
            
            # First layer
            first = get_int("Neurons in first hidden layer", 4, 4096, 128)
            layers.append(first)
            
            print(f"\n{Colors.CYAN}Enter layers 2-3 in descending order (optional){Colors.ENDC}")
            
            suggestions = [64, 32]
            for i in range(2):
                cont = input(f"{Colors.CYAN}Add layer {i+2}? (y/n, suggested: {suggestions[i]}): {Colors.ENDC}").strip().lower()
                if cont == 'n':
                    break
                
                while True:
                    neurons = get_int(f"Neurons for layer {i+2}", 4, 4096, suggestions[i])
                    if neurons == 2:
                        Colors.error("Cannot use 2 neurons")
                    elif neurons >= layers[-1]:
                        Colors.error(f"Must be less than previous layer ({layers[-1]})")
                    else:
                        layers.append(neurons)
                        break
            
            # Additional layers
            if len(layers) >= 3:
                while True:
                    cont = input(f"{Colors.CYAN}Add another layer? (y/n): {Colors.ENDC}").strip().lower()
                    if cont == 'n':
                        break
                    
                    while True:
                        neurons = get_int("Neurons for layer", 4, 4096, layers[-1]//2)
                        if neurons == 2:
                            Colors.error("Cannot use 2 neurons")
                        elif neurons >= layers[-1]:
                            Colors.error(f"Must be less than {layers[-1]}")
                        else:
                            layers.append(neurons)
                            break
            
            return tuple(layers)
        
        params = {
            'hidden_layers': get_hidden_layers(),
            'learning_rate': get_float("Learning rate", 0.00001, 0.5, 0.001),
            'epochs': get_int("Number of epochs", 50, 10000, 1000),
            'batch_size': get_int("Batch size", 2, 512, 256),
            'random_state': get_int("Random state", 2, 314, 42)
        }
        
        print(f"\n{Colors.BOLD}Selected Hyperparameters:{Colors.ENDC}")
        for key, val in params.items():
            print(f"  {key}: {val}")
        
        return params
    
    @staticmethod
    def run_grid_search(X_train, y_train, input_dim, random_state=42) -> Tuple[Dict, object, pd.DataFrame]:
        """Run GridSearchCV for automatic hyperparameter tuning"""
        Colors.step(3, "AUTOMATIC HYPERPARAMETER TUNING (GridSearchCV)")
        
        max_cpu = os.cpu_count()
        while True:
            try:
                n_jobs = int(input(f"{Colors.CYAN}CPUs to use (-1 for all, max {max_cpu}): {Colors.ENDC}"))
                if n_jobs == -1 or (0 < n_jobs <= max_cpu):
                    break
                Colors.error(f"Enter -1 or 1-{max_cpu}")
            except ValueError:
                Colors.error("Enter a valid integer")
        
        def create_model(hidden_layers=(128, 64), learning_rate=0.001, input_dim=None):
            return DNNBuilder.build_model(input_dim, hidden_layers if isinstance(hidden_layers, tuple) else (hidden_layers,), learning_rate)
        
        param_grid = {
            'model__hidden_layers': [
                (8,), (16,), (32,), (64,), (128,), (256,),
                (128, 64), (256, 128), (512, 256),
                (512, 256, 128), (1024, 512, 256)
            ],
            'model__learning_rate': [0.0001, 0.001, 0.01],
            'batch_size': [32, 64, 128, 256],
            'epochs': [1000, 1500, 2000]
        }
        
        scorings = {
            'R2': make_scorer(r2_score, greater_is_better=True),
            'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
            'RMSE': make_scorer(root_mean_squared_error, greater_is_better=False),
            'MaxError': make_scorer(max_error, greater_is_better=False)
        }
        
        model_wrapper = KerasRegressor(
            model=create_model,
            model__input_dim=input_dim,
            verbose=0
        )
        
        grid = GridSearchCV(
            estimator=model_wrapper,
            param_grid=param_grid,
            scoring=scorings,
            refit='R2',
            cv=KFold(n_splits=5, shuffle=True, random_state=random_state),
            n_jobs=n_jobs,
            verbose=3,
            return_train_score=True
        )
        
        Colors.info("Starting Grid Search (this may take a while)...")
        start = time.time()
        grid.fit(X_train, y_train)
        elapsed = time.time() - start
        
        Colors.success(f"Grid Search completed in {elapsed:.2f}s")
        
        cv_results = pd.DataFrame(grid.cv_results_)
        error_metrics = [col for col in cv_results.columns if any(err in col for err in ["MAE", "RMSE", "MaxError"])]
        cv_results[error_metrics] = cv_results[error_metrics].abs()
        
        best_params = {
            'hidden_layers': grid.best_params_['model__hidden_layers'],
            'learning_rate': grid.best_params_['model__learning_rate'],
            'epochs': grid.best_params_['epochs'],
            'batch_size': grid.best_params_['batch_size']
        }
        
        print(f"\n{Colors.BOLD}Best Parameters:{Colors.ENDC}")
        for key, val in best_params.items():
            print(f"  {key}: {val}")
        print(f"\n{Colors.BOLD}Best R² Score: {grid.best_score_:.5f}{Colors.ENDC}")
        
        return best_params, grid, cv_results

# ============================================================================
# PLOT DATA STORE
# ============================================================================
class PlotDataStore:
    @staticmethod
    def save(save_dir, save_name,
             y_train_actual, y_train_pred, y_test_actual, y_test_pred,
             train_loss, val_loss, cv_scores, model_info, output_latex,
             train_metrics, test_metrics):

        npz_path  = os.path.join(save_dir, f"{save_name}_plot_data.npz")
        json_path = os.path.join(save_dir, f"{save_name}_plot_meta.json")

        np.savez_compressed(
            npz_path,
            y_train_actual=y_train_actual.flatten(),
            y_train_pred=y_train_pred.flatten(),
            y_test_actual=y_test_actual.flatten(),
            y_test_pred=y_test_pred.flatten(),
            train_loss=np.array(train_loss),
            val_loss=np.array(val_loss),
            cv_r2=np.array(cv_scores['R2']),
            cv_mae=np.array(cv_scores['MAE']),
            cv_rmse=np.array(cv_scores['RMSE']),
            cv_maxerr=np.array(cv_scores['MaxError']),
        )

        def _to_serializable(v):
            if isinstance(v, tuple):
                return list(v)
            if isinstance(v, (np.integer,)):
                return int(v)
            if isinstance(v, (np.floating,)):
                return float(v)
            return v

        meta = {
            'output_latex': output_latex,
            'model_info':   {k: _to_serializable(v) for k, v in model_info.items()},
            'train_metrics': {k: float(v) for k, v in train_metrics.items()},
            'test_metrics':  {k: float(v) for k, v in test_metrics.items()},
        }
        with open(json_path, 'w') as f:
            json.dump(meta, f, indent=4)

        Colors.success(f"Plot data saved: {npz_path}")

    @staticmethod
    def load(save_dir, save_name):
        npz_path  = os.path.join(save_dir, f"{save_name}_plot_data.npz")
        json_path = os.path.join(save_dir, f"{save_name}_plot_meta.json")

        if not os.path.exists(npz_path) or not os.path.exists(json_path):
            return None, None

        arrays = dict(np.load(npz_path, allow_pickle=False))
        with open(json_path, 'r') as f:
            meta = json.load(f)

        if 'hidden_layers' in meta.get('model_info', {}):
            meta['model_info']['hidden_layers'] = tuple(meta['model_info']['hidden_layers'])

        Colors.success(f"Plot data loaded from: {npz_path}")
        return arrays, meta

    @staticmethod
    def regenerate_figures(save_dir, save_name):
        """Call this to regenerate all figures without retraining."""
        arrays, meta = PlotDataStore.load(save_dir, save_name)
        if arrays is None:
            Colors.error("No plot data found. Train the model first.")
            return

        output_latex  = meta['output_latex']
        model_info    = meta['model_info']
        train_metrics = meta['train_metrics']
        test_metrics  = meta['test_metrics']

        y_train_actual = arrays['y_train_actual'].reshape(-1, 1)
        y_train_pred   = arrays['y_train_pred'].reshape(-1, 1)
        y_test_actual  = arrays['y_test_actual'].reshape(-1, 1)
        y_test_pred    = arrays['y_test_pred'].reshape(-1, 1)
        cv_scores = {
            'R2':       arrays['cv_r2'].tolist(),
            'MAE':      arrays['cv_mae'].tolist(),
            'RMSE':     arrays['cv_rmse'].tolist(),
            'MaxError': arrays['cv_maxerr'].tolist(),
        }

        Visualizer.plot_actual_vs_predicted(
            y_train_actual, y_train_pred, y_test_actual, y_test_pred,
            model_info, output_latex, save_dir, save_name,
            r2_train=train_metrics['R2'],       r2_test=test_metrics['R2'],
            mae_train=train_metrics['MAE'],     mae_test=test_metrics['MAE'],
            rmse_train=train_metrics['RMSE'],   rmse_test=test_metrics['RMSE'],
            max_err_train=train_metrics['MaxError'], max_err_test=test_metrics['MaxError'],
        )
        Visualizer.plot_training_history(
            arrays['train_loss'].tolist(),
            arrays['val_loss'].tolist(),
            output_latex, save_dir, save_name
        )
        Visualizer.plot_cv_results(cv_scores, output_latex, save_dir, save_name)
        Colors.success("All figures regenerated.")
        
# ============================================================================
# VISUALIZER
# ============================================================================
class Visualizer:
    """Create professional visualizations"""
    
    @staticmethod
    def plot_actual_vs_predicted(y_train_actual, y_train_pred, y_test_actual, y_test_pred,
                                  model_info, output_latex, save_dir, save_name,
                                  r2_train, r2_test, mae_train, mae_test,
                                  rmse_train, rmse_test, max_err_train, max_err_test):
        plt.figure(figsize=(8, 5))
        
        # Training data
        plt.scatter(y_train_actual.flatten(), y_train_pred.flatten(),
                   s=80, alpha=0.6, c='red', marker='s',
                   edgecolors='none', label='Training', zorder=2)
        
        # Test data
        plt.scatter(y_test_actual.flatten(), y_test_pred.flatten(),
                   s=80, alpha=0.7, c='#00008B', marker='s',
                   edgecolors='none', label='Test', zorder=3)
        
        # Ideal fit line
        min_val = min(y_train_actual.min(), y_test_actual.min(),
                     y_train_pred.min(), y_test_pred.min())
        max_val = max(y_train_actual.max(), y_test_actual.max(),
                     y_train_pred.max(), y_test_pred.max())
        
        padding = 0.05 * (max_val - min_val)
        plt.plot([min_val-padding, max_val+padding], [min_val-padding, max_val+padding],
                'k--', lw=2, alpha=0.8, label='Ideal Fit', zorder=1)
        
        # Max error point
        y_all_actual = np.concatenate([y_train_actual.flatten(), y_test_actual.flatten()])
        y_all_pred = np.concatenate([y_train_pred.flatten(), y_test_pred.flatten()])
        errors = np.abs(y_all_actual - y_all_pred)
        max_idx = np.argmax(errors)
        
        plt.scatter(y_all_actual[max_idx], y_all_pred[max_idx],
                   s=200, c='blue', marker='x', linewidths=3,
                   edgecolors='black', label='Max Error', zorder=5)
        
        # Formatting
        ax = plt.gca()
        ax.set_xlim(min_val-padding, max_val+padding)
        ax.set_ylim(min_val-padding, max_val+padding)
        
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.tick_params(axis='both', which='major', direction='in', length=8,
                      labelsize=20, top=True, right=True, left=True, bottom=True)
        ax.tick_params(axis='both', which='minor', direction='in', length=3,
                      top=True, right=True, left=True, bottom=True)
        ax.minorticks_on()
        ax.xaxis.set_minor_locator(AutoMinorLocator(13))
        ax.yaxis.set_minor_locator(AutoMinorLocator(10))

        
        plt.xlabel(fr"Actual {output_latex}", fontsize=24)
        plt.ylabel(fr"Predicted {output_latex}", fontsize=24)
        # ----------------------------------------------------------------
        # Metrics Annotation Box (R², MAE, RMSE)
        # ----------------------------------------------------------------
        plt.annotate(
            f"$R^2$ (Train): {r2_train:.4f}\n"
            f"$R^2$ (Test): {r2_test:.4f}\n"
            f"MAE (Train): {mae_train:.3f}\n"
            f"MAE (Test): {mae_test:.3f}\n"
            f"RMSE (Train): {rmse_train:.3f}\n"
            f"RMSE (Test): {rmse_test:.3f}",
            xy=(0.57, 0.37), xycoords='axes fraction',
            fontsize=12.5, ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', alpha=0.8)
        )
    
        # ----------------------------------------------------------------
        # Max Error Annotation Box
        # ----------------------------------------------------------------
        plt.annotate(
            f"Max Error (Train): {max_err_train:.3f}\n"
            f"Max Error (Test): {max_err_test:.3f}",
            xy=(0.053, 0.938), xycoords='axes fraction',
            fontsize=12.5, ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', alpha=0.8)
        )

        # ----------------------------------------------------------------
        # Model info box
        # ----------------------------------------------------------------
        box_text = '\n'.join([
            'DNN Model Information',
            '',
            f"Input Dim: {model_info['input_dim']}",
            f"Total samples: {model_info['n_total']}",
            f"Train: {model_info['n_train']}",
            f"Test: {model_info['n_test']}",
            f"Total layers: {model_info['n_layers']}",
            f"Neurons: {model_info['n_neurons']}",
            f"Parameters: {model_info['params']:,}",
            f"Hidden layers: {model_info['hidden_layers']}",
            f"LR: {model_info['lr']}",
            f"Epochs: {model_info['epochs']}",
            f"Batch: {model_info['batch']}"
        ])
        
        box_props = dict(boxstyle='round,pad=0.4', edgecolor='black', facecolor='white', alpha=1)
        plt.annotate(box_text, xy=(1.05, 0.5), xycoords='axes fraction',
                    fontsize=12, bbox=box_props, verticalalignment='center')
        
        plt.legend(fontsize=13, frameon=False, loc='upper center',
                  bbox_to_anchor=(0.5, 1.12), ncol=4)
        
        plt.tight_layout()
        plt.grid(False)
        for fmt in ['png', 'pdf']:
            save_path = os.path.join(save_dir, f"{save_name}_Actual_vs_Predicted.{fmt}")
            plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()
    
    @staticmethod
    def plot_training_history(train_loss, val_loss, output_latex, save_dir, save_name):
        loss = train_loss
        plt.figure(figsize=(14, 5))
        
        # Training loss
        plt.plot(range(len(loss)), loss, '-', color='darkblue',
                lw=1.5, label='Training')
        
        # Validation loss
        if val_loss:
            plt.plot(range(len(val_loss)), val_loss, '-', color='black',
                    lw=1.5, label='Validation')
        
        plt.ylim(-0.01, 0.3)
        
        plt.xlabel('Epoch', fontsize=20, fontweight='bold')
        plt.ylabel(fr'Loss {output_latex}', fontsize=20, fontweight='bold')
        
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(1.5)
            spine.set_color('black')
        
        ax.minorticks_on()
        plt.tick_params(axis='both', which='major', labelsize=20, direction='in',
                       length=10, width=1.5, top=True, right=True, left=True, bottom=True)
        plt.tick_params(axis='both', which='minor', direction='in',
                       length=5, width=1, top=True, right=True, left=True, bottom=True)
        ax.xaxis.set_minor_locator(AutoMinorLocator(15))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        
        plt.legend(fontsize=20, frameon=False)
        plt.grid(False)
        plt.tight_layout()
        for fmt in ['png', 'pdf']:
            save_path = os.path.join(save_dir, f"{save_name}_Loss_vs_Epoch.{fmt}")
            plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()
    
    @staticmethod
    def plot_cv_results(cv_scores, output_latex, save_dir, save_name):
        # Plot 5-fold cross-validation results
        folds = np.arange(1, 6)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
        
        colors = {
            'r2': '#0343DF',
            'mae': '#2ca02c',
            'rmse': 'red',
            'max_err': 'mediumslateblue'
        }
        
        configs = [
            (ax1, cv_scores['R2'], fr'R² Score {output_latex}', colors['r2']),
            (ax2, cv_scores['MAE'], fr'MAE {output_latex}', colors['mae']),
            (ax3, cv_scores['RMSE'], fr'RMSE {output_latex}', colors['rmse']),
            (ax4, cv_scores['MaxError'], fr'Max Error {output_latex}', colors['max_err'])
        ]
        
        for ax, scores, ylabel, color in configs:
            ax.bar(folds, scores, width=0.5, color=color,
                  edgecolor='black', linewidth=1.5, alpha=0.85)
            
            mean_val = np.mean(scores)
            ax.axhline(mean_val, color='darkred', linestyle='--',
                      linewidth=3, label=f'Mean: {mean_val:.4f}', zorder=10)
            
            ax.set_xlabel('Fold Number', fontsize=20, fontweight='bold')
            ax.set_ylabel(ylabel, fontsize=20, fontweight='bold')
            
            max_val = np.max(scores)
            min_val = np.min(scores)
            data_range = max_val - min_val
            y_upper = max_val + 0.5 * data_range
            y_lower = min_val - 0.05 * data_range
            ax.set_ylim(y_lower, y_upper)
            
            legend = ax.legend(fontsize=20, frameon=True, loc='best')
            legend.get_frame().set_edgecolor('black')
            legend.get_frame().set_linewidth(1.5)
            
            ax.set_xticks(folds)
            ax.set_xticklabels(folds, fontsize=13)
            ax.tick_params(axis='y', labelsize=13)
            
            for spine in ax.spines.values():
                spine.set_linewidth(1.5)
                spine.set_color('black')
            
            ax.minorticks_on()
            ax.tick_params(axis='both', which='major', direction='in',
                          length=8, width=1.5, right=True, left=True, labelsize=20)
            ax.tick_params(axis='both', which='minor', direction='in',
                          length=4, width=1.5, right=True, left=True)
        
        plt.tight_layout()
        plt.grid(False)
        for fmt in ['png', 'pdf']:
            save_path = os.path.join(save_dir, f"{save_name}_5_Fold_CV_Plot.{fmt}")
            plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()
    
    @staticmethod
    def plot_grid_search_results(cv_results, save_dir, save_name):
        # Plot grid search metric evolution
        metrics = ['mean_test_R2', 'mean_test_MAE', 'mean_test_RMSE', 'mean_test_MaxError']
        labels = ['R² Score', 'MAE', 'RMSE', 'Max Error']
        higher_is_better = [True, False, False, False]
        
        for metric, label, is_higher in zip(metrics, labels, higher_is_better):
            plt.figure(figsize=(10, 6))
            
            plt.plot(cv_results.index + 1, cv_results[metric],
                    marker='o', linestyle='-', linewidth=2,
                    markersize=8, markeredgecolor='black', markeredgewidth=1.5,
                    markerfacecolor='royalblue', color='black')
            
            if is_higher:
                best_idx = cv_results[metric].idxmax()
                best_val = cv_results[metric].max()
            else:
                best_idx = cv_results[metric].idxmin()
                best_val = cv_results[metric].min()
            
            plt.axhline(best_val, color='red', linestyle='--', linewidth=2,
                       label=f'Best {label}: {best_val:.4f}')
            plt.axvline(best_idx + 1, color='red', linestyle='--', linewidth=2)
            
            plt.xlabel("Grid Search Iteration", fontsize=14)
            plt.ylabel(label, fontsize=14)
            plt.legend(fontsize=12)
            plt.grid(False)
            plt.tight_layout()
            
            for fmt in ['png', 'pdf']:
                save_path = os.path.join(save_dir, f"{save_name}_{label.replace(' ', '_')}.{fmt}")
                plt.savefig(save_path, dpi=600, bbox_inches='tight')
            plt.close()

# ============================================================================
# RESULTS EXPORTER
# ============================================================================
class ResultsExporter:
    """Export all results to Excel and save models"""
    
    def __init__(self, save_dir: str, save_name: str):
        self.save_dir = save_dir
        self.save_name = save_name
        self.model_folder = os.path.join(save_dir, save_name)
        os.makedirs(self.model_folder, exist_ok=True)
        self._results_path = os.path.join(save_dir, f"{save_name}_Optimized_DNN_Results.xlsx")
    
    def _write_sheet(self, df: pd.DataFrame, sheet_name: str):
        if os.path.exists(self._results_path):
            with pd.ExcelWriter(self._results_path, engine='openpyxl', mode='a',
                                if_sheet_exists='replace') as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            with pd.ExcelWriter(self._results_path, engine='openpyxl', mode='w') as writer:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
            
    def save_model_and_scalers(self, model, scaler_X, scaler_y):
        """Save trained model and scalers"""
        model_path = os.path.join(self.model_folder, f"{self.save_name}_best_model.keras")
        
        if hasattr(model, 'model_'):
            model.model_.save(model_path)
        else:
            model.save(model_path)
        
        joblib.dump(scaler_X, os.path.join(self.model_folder, f'{self.save_name}_scaler_X.joblib'))
        joblib.dump(scaler_y, os.path.join(self.model_folder, f'{self.save_name}_scaler_y.joblib'))
        
        Colors.success(f"Model saved: {self.model_folder}")
    
    def save_metadata(self, X, y, scaler_X, scaler_y, trainable_params,
                     best_params, train_metrics, test_metrics):
        """Save model metadata as JSON"""
        metadata = {
            "model_name": "DNN_Best_Model",
            "input_features": list(X.columns),
            "output_features": [y.name],
            "trainable_parameters": int(trainable_params),
            "best_params": {k: (str(v) if isinstance(v, tuple) else v) 
                          for k, v in best_params.items()},
            "scaler_X_mean": scaler_X.mean_.tolist(),
            "scaler_X_scale": scaler_X.scale_.tolist(),
            "scaler_y_mean": scaler_y.mean_.tolist(),
            "scaler_y_scale": scaler_y.scale_.tolist(),
            "scaler_X_min": X.min(axis=0).tolist(),
            "scaler_X_max": X.max(axis=0).tolist(),
            "scaler_y_min": float(y.min()),
            "scaler_y_max": float(y.max()),
            "train_metrics": {k: float(v) for k, v in train_metrics.items()},
            "test_metrics": {k: float(v) for k, v in test_metrics.items()}
        }
        
        path = os.path.join(self.model_folder, f"{self.save_name}_metadata.json")
        with open(path, 'w') as f:
            json.dump(metadata, f, indent=4)
        
        Colors.success(f"Metadata saved: {path}")
    
    def save_predictions(self, y_train_actual, y_train_pred, y_test_actual, y_test_pred):
        train_df = pd.DataFrame({
            'Actual_Train': y_train_actual.flatten(),
            'Predicted_Train': y_train_pred.flatten()
        })
        test_df = pd.DataFrame({
            'Actual_Test': y_test_actual.flatten(),
            'Predicted_Test': y_test_pred.flatten()
        })
        self._write_sheet(train_df, 'Predictions_Train')
        self._write_sheet(test_df,  'Predictions_Test')
        Colors.success("Sheet written: Predictions_Train / Predictions_Test")
    
    def save_model_details(self, config_data, architecture_data, param_breakdown,
                           hyperparams, performance_metrics):
        self._write_sheet(pd.DataFrame(config_data),         'Model_Configuration')
        self._write_sheet(pd.DataFrame(architecture_data),   'Architecture')
        self._write_sheet(pd.DataFrame(param_breakdown),     'Parameter_Breakdown')
        self._write_sheet(pd.DataFrame(hyperparams),         'Hyperparameters')
        self._write_sheet(pd.DataFrame(performance_metrics), 'Performance_Metrics')
        Colors.success("Sheets written: model detail sheets → Results workbook")
    
    def save_cv_results(self, cv_results_df):
        self._write_sheet(cv_results_df, '5_Fold_CV_Results')
        Colors.success("Sheet written: 5_Fold_CV_Results")
    
    def save_training_history(self, history):
        loss     = history.history['loss']
        val_loss = history.history.get('val_loss', [None] * len(loss))
        loss_df  = pd.DataFrame({
            "Epoch": list(range(1, len(loss) + 1)),
            "Training_Loss": loss,
            "Validation_Loss": val_loss
        })
        self._write_sheet(loss_df, 'Loss_vs_Epoch')
        Colors.success("Sheet written: Loss_vs_Epoch")
    
    def save_timing_data(self, timing_data):
        self._write_sheet(pd.DataFrame(timing_data), 'Timing_Summary')
        Colors.success(f"Sheet written: Timing_Summary  →  {self._results_path}")

    def save_grid_search_results(self, cv_results: pd.DataFrame, best_params: Dict,
                                  best_score: float):
        gs_path = os.path.join(self.save_dir, f"{self.save_name}_GridSearch.xlsx")
        with pd.ExcelWriter(gs_path, engine='openpyxl', mode='w') as writer:
            cv_results.to_excel(writer, sheet_name='CV_Results', index=True)
            summary = pd.DataFrame({
                'Parameter': list(best_params.keys()) + ['Best_CV_R2'],
                'Value':     [str(v) for v in best_params.values()] + [f"{best_score:.6f}"]
            })
            summary.to_excel(writer, sheet_name='Best_Params', index=False)
        Colors.success(f"Grid Search workbook saved: {gs_path}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution flow"""
    
    # Initialize timing
    timing_data = {'Operation': [], 'Time (seconds)': []}
    start_total = time.time()
    
    Colors.header("DEEP NEURAL NETWORK TRAINING SYSTEM v2.0")
    
    # Step 1: Load data
    file_path, sheet_name = FileManager.select_data_file()
    df = DataLoader.load_file(file_path, sheet_name)
    
    if df is None:
        sys.exit(1)
    
    # Step 2: Select columns
    X, y, output_latex = DataLoader.select_columns(df)
    
    # Validate data dimensions
    n_samples, n_features = X.shape
    if n_features > n_samples:
        Colors.error(f"Features ({n_features}) exceed samples ({n_samples})!")
        sys.exit(1)
    
    Colors.success("Data validation passed")
    
    # Step 3: Setup output
    save_dir, save_name = FileManager.setup_output_directory()
    exporter = ResultsExporter(save_dir, save_name)
    
    # Step 4: Process data
    test_size = DataProcessor.determine_test_size(n_samples)
    processor = DataProcessor(test_size=test_size, random_state=42)
    
    Colors.info(f"Data points: {n_samples}")
    Colors.info(f"Test size: {test_size*100:.0f}%")
    
    X_train, X_test, y_train, y_test = processor.split_data(X.values, y.values.reshape(-1, 1))

    X_train = processor.scaler_X.fit_transform(X_train)
    X_test = processor.scaler_X.transform(X_test)
    y_train = processor.scaler_y.fit_transform(y_train)
    y_test = processor.scaler_y.transform(y_test)
    
    # Keep X_scaled and y_scaled for post-hoc CV (fit already done on train)
    X_scaled = processor.scaler_X.transform(X.values)
    y_scaled = processor.scaler_y.transform(y.values.reshape(-1, 1))

    # Step 5: Configure hyperparameters
    while True:
        mode = input(f"\n{Colors.CYAN}Hyperparameter mode (manual/auto): {Colors.ENDC}").strip().lower()
        
        if mode == 'manual':
            # Manual mode
            params = HyperparameterManager.get_manual_params()
            
            Colors.step(4, "TRAINING DNN MODEL")
            start_train = time.time()
            
            model = DNNBuilder.build_model(
                X_train.shape[1],
                params['hidden_layers'],
                params['learning_rate']
            )
            
            reduce_lr = callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, 
                patience=20, min_lr=1e-6)
            
            history = model.fit(
                X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=params['epochs'],
                batch_size=params['batch_size'],
                callbacks=[reduce_lr],
                verbose=1
            )

            elapsed_train = time.time() - start_train
            timing_data['Operation'].append('Model Training')
            timing_data['Time (seconds)'].append(elapsed_train)
            
            Colors.success(f"Training completed in {elapsed_train:.2f}s")
            
            best_model = model
            best_params = params
            break
        
        elif mode == 'auto':
            # Auto mode with GridSearch
            best_params, grid, cv_results = HyperparameterManager.run_grid_search(
                X_train, y_train, X_train.shape[1], random_state=42
            )
            exporter.save_grid_search_results(cv_results, best_params, grid.best_score_)
            
            # Plot grid search results
            Visualizer.plot_grid_search_results(cv_results, save_dir, save_name)
            
            # Retrain best model to get history
            Colors.step(4, "RETRAINING BEST MODEL")
            start_train = time.time()
            
            best_model = DNNBuilder.build_model(
                X_train.shape[1],
                best_params['hidden_layers'],
                best_params['learning_rate']
            )
            reduce_lr = callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=20, min_lr=1e-6)
            
            history = best_model.fit(
                X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=best_params['epochs'],
                batch_size=best_params['batch_size'],
                callbacks=[reduce_lr],
                verbose=1
            )
            
            elapsed_train = time.time() - start_train
            timing_data['Operation'].append('Model Retraining')
            timing_data['Time (seconds)'].append(elapsed_train)
            
            Colors.success(f"Retraining completed in {elapsed_train:.2f}s")
            
            break
        else:
            Colors.error("Enter 'manual' or 'auto'")
    
    # Step 6: Evaluate model
    Colors.step(5, "EVALUATING MODEL PERFORMANCE")
    
    start_pred = time.time()
    y_train_pred = best_model.predict(X_train, verbose=0)
    elapsed_pred_train = time.time() - start_pred
    timing_data['Operation'].append('Training Set Prediction')
    timing_data['Time (seconds)'].append(elapsed_pred_train)
    
    start_pred = time.time()
    y_test_pred = best_model.predict(X_test, verbose=0)
    elapsed_pred_test = time.time() - start_pred
    timing_data['Operation'].append('Test Set Prediction')
    timing_data['Time (seconds)'].append(elapsed_pred_test)
    
    # Inverse transform
    y_train_actual = processor.scaler_y.inverse_transform(y_train)
    y_train_pred_orig = processor.scaler_y.inverse_transform(y_train_pred.reshape(-1, 1))
    y_test_actual = processor.scaler_y.inverse_transform(y_test)
    y_test_pred_orig = processor.scaler_y.inverse_transform(y_test_pred.reshape(-1, 1))
    
    # Calculate metrics
    train_metrics = MetricsCalculator.calculate_all(y_train_actual, y_train_pred_orig)
    test_metrics = MetricsCalculator.calculate_all(y_test_actual, y_test_pred_orig)
    
    print(f"\n{Colors.BOLD}Training Metrics:{Colors.ENDC}")
    for metric, value in train_metrics.items():
        print(f"  {metric}: {value:.5f}")
    
    print(f"\n{Colors.BOLD}Test Metrics:{Colors.ENDC}")
    for metric, value in test_metrics.items():
        print(f"  {metric}: {value:.5f}")
    
    # Step 7: Calculate parameters
    trainable_params, param_breakdown = DNNBuilder.calculate_parameters(
        best_model.model_ if hasattr(best_model, 'model_') else best_model
    )
    
    Colors.info(f"Total trainable parameters: {trainable_params:,}")
    
    # Step 8: Save model and results
    Colors.step(6, "SAVING RESULTS")
    
    exporter.save_model_and_scalers(best_model, processor.scaler_X, processor.scaler_y)
    exporter.save_metadata(X, y, processor.scaler_X, processor.scaler_y,
                          trainable_params, best_params, train_metrics, test_metrics)
    exporter.save_predictions(y_train_actual, y_train_pred_orig,
                             y_test_actual, y_test_pred_orig)
    
    # Model info
    n_train, n_test = len(y_train), len(y_test)
    total_layers = 1 + len(best_params['hidden_layers']) + 1
    total_neurons = sum(best_params['hidden_layers'])
    
    config_data = {
        'Configuration': ['Model Type', 'Input Dimension', 'Output Dimension',
                         'Total Samples', 'Training Samples', 'Test Samples',
                         'Test Size Ratio', 'Random State'],
        'Values': ['DNN', n_features, 1, n_samples, n_train, n_test,
                  test_size, 42]
    }
    
    architecture_data = {
        'Architecture': ['Total Layers', 'Total Hidden Neurons',
                        'Hidden Layer Config', 'Trainable Parameters',
                        'Activation', 'Initializer', 'Regularizer'],
        'Values': [total_layers, total_neurons, str(best_params['hidden_layers']),
                  trainable_params, 'PReLU', 'glorot_uniform', 'L2(0.0001)']
    }
    
    param_breakdown_data = {
        'Layer': [],
        'Weights': [],
        'Biases': [],
        'Total': []
    }
    
    for layer_name, params in param_breakdown.items():
        param_breakdown_data['Layer'].append(layer_name)
        param_breakdown_data['Weights'].append(params['weights'])
        param_breakdown_data['Biases'].append(params['biases'])
        param_breakdown_data['Total'].append(params['total'])
    
    param_breakdown_data['Layer'].append('TOTAL')
    param_breakdown_data['Weights'].append(sum(p['weights'] for p in param_breakdown.values()))
    param_breakdown_data['Biases'].append(sum(p['biases'] for p in param_breakdown.values()))
    param_breakdown_data['Total'].append(trainable_params)
    
    hyperparams_data = {
        'Hyperparameter': ['Learning Rate', 'Optimizer', 'Weight Decay',
                          'Epochs', 'Batch Size', 'Loss Function'],
        'Values': [best_params['learning_rate'], 'AdamW', '1e-5',
                  best_params['epochs'], best_params['batch_size'], 'MSE']
    }
    
    performance_data = {
        'Metric': ['R² Score', 'MAE', 'RMSE', 'Max Error',
                  'Inference Time (s)'],
        'Training': [f"{train_metrics['R2']:.6f}",
                    f"{train_metrics['MAE']:.6f}",
                    f"{train_metrics['RMSE']:.6f}",
                    f"{train_metrics['MaxError']:.6f}",
                    f"{elapsed_pred_train:.6f}"],
        'Test': [f"{test_metrics['R2']:.6f}",
                f"{test_metrics['MAE']:.6f}",
                f"{test_metrics['RMSE']:.6f}",
                f"{test_metrics['MaxError']:.6f}",
                f"{elapsed_pred_test:.6f}"]
    }
    
    exporter.save_model_details(config_data, architecture_data,
                               param_breakdown_data, hyperparams_data,
                               performance_data)
    
    # Step 9: Create visualizations
    Colors.step(7, "CREATING VISUALIZATIONS")
    
    model_info = {
        'input_dim': n_features,
        'n_total': n_samples,
        'n_train': n_train,
        'n_test': n_test,
        'n_layers': total_layers,
        'n_neurons': total_neurons,
        'params': trainable_params,
        'hidden_layers': best_params['hidden_layers'],
        'lr': best_params['learning_rate'],
        'epochs': best_params['epochs'],
        'batch': best_params['batch_size']
    }
    
    Visualizer.plot_actual_vs_predicted(
        y_train_actual, y_train_pred_orig,
        y_test_actual, y_test_pred_orig,
        model_info, output_latex,
        save_dir, save_name,          
        r2_train=train_metrics['R2'],
        r2_test=test_metrics['R2'],
        mae_train=train_metrics['MAE'],
        mae_test=test_metrics['MAE'],
        rmse_train=train_metrics['RMSE'],
        rmse_test=test_metrics['RMSE'],
        max_err_train=train_metrics['MaxError'],
        max_err_test=test_metrics['MaxError']
    )
    
    Visualizer.plot_training_history(
        history.history['loss'],
        history.history.get('val_loss', []),
        output_latex, save_dir, save_name
    )
    exporter.save_training_history(history)
    
    # Step 10: Cross-validation
    Colors.step(8, "PERFORMING 5-FOLD CROSS-VALIDATION")
    
    start_cv = time.time()
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    cv_scores = {'R2': [], 'MAE': [], 'RMSE': [], 'MaxError': []}
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_scaled), 1):
        X_train_cv = X_scaled[train_idx]
        X_test_cv = X_scaled[test_idx]
        y_train_cv = y_scaled[train_idx]
        y_test_cv = y_scaled[test_idx]
        fold_model = DNNBuilder.build_model(
            X_train_cv.shape[1],
            best_params['hidden_layers'],
            best_params['learning_rate']
        )

        reduce_lr_cv = callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=20, min_lr=1e-6)

        fold_model.fit(
            X_train_cv, y_train_cv,
            validation_data=(X_test_cv, y_test_cv),
            epochs=best_params['epochs'],
            batch_size=best_params['batch_size'],
            callbacks=[reduce_lr_cv],
            verbose=0
        )
        
        y_pred_cv = fold_model.predict(X_test_cv, verbose=0)
        
        y_test_cv_orig = processor.scaler_y.inverse_transform(y_test_cv)
        y_pred_cv_orig = processor.scaler_y.inverse_transform(y_pred_cv.reshape(-1, 1))
        
        metrics_cv = MetricsCalculator.calculate_all(y_test_cv_orig, y_pred_cv_orig)
        
        for key in cv_scores:
            cv_scores[key].append(metrics_cv[key])
        
        print(f"Fold {fold}: R²={metrics_cv['R2']:.4f}, MAE={metrics_cv['MAE']:.4f}, "
              f"RMSE={metrics_cv['RMSE']:.4f}, MaxErr={metrics_cv['MaxError']:.4f}")
    
    elapsed_cv = time.time() - start_cv
    timing_data['Operation'].append('5-Fold CV')
    timing_data['Time (seconds)'].append(elapsed_cv)
    
    # Save CV results
    cv_results_df = pd.DataFrame({
        'Fold': np.arange(1, 6),
        'R² Score': cv_scores['R2'],
        'MAE': cv_scores['MAE'],
        'RMSE': cv_scores['RMSE'],
        'Max Error': cv_scores['MaxError']
    })
    
    cv_results_df.loc['Mean'] = ['Mean'] + [np.mean(cv_scores[k]) for k in ['R2', 'MAE', 'RMSE', 'MaxError']]
    cv_results_df.loc['Std'] = ['Std'] + [np.std(cv_scores[k]) for k in ['R2', 'MAE', 'RMSE', 'MaxError']]
    
    exporter.save_cv_results(cv_results_df)
    PlotDataStore.save(
        save_dir, save_name,
        y_train_actual, y_train_pred_orig,
        y_test_actual,  y_test_pred_orig,
        train_loss=history.history['loss'],
        val_loss=history.history.get('val_loss', []),
        cv_scores=cv_scores,
        model_info=model_info,
        output_latex=output_latex,
        train_metrics=train_metrics,
        test_metrics=test_metrics,
    )
    
    # Plot CV results
    Visualizer.plot_cv_results(
        cv_scores, output_latex,
        save_dir, save_name
    )
    
    # Timing
    elapsed_total = time.time() - start_total
    timing_data['Operation'].append('Total Runtime')
    timing_data['Time (seconds)'].append(elapsed_total)
    
    exporter.save_timing_data(timing_data)
    
    # Final summary
    Colors.header("TRAINING COMPLETE")
    
    print(f"\n{Colors.BOLD}Best Model Configuration:{Colors.ENDC}")
    for key, val in best_params.items():
        print(f"  {key}: {val}")
    
    print(f"\n{Colors.BOLD}Test Performance:{Colors.ENDC}")
    for key, val in test_metrics.items():
        print(f"  {key}: {val:.5f}")
    
    print(f"\n{Colors.BOLD}5-Fold CV Performance:{Colors.ENDC}")
    for key in ['R2', 'MAE', 'RMSE', 'MaxError']:
        mean_val = np.mean(cv_scores[key])
        std_val = np.std(cv_scores[key])
        print(f"  {key}: {mean_val:.5f} ± {std_val:.5f}")
    
    print(f"\n{Colors.BOLD}Timing Summary:{Colors.ENDC}")
    for op, t in zip(timing_data['Operation'], timing_data['Time (seconds)']):
        print(f"  {op}: {t:.2f}s")
    
    print(f"\n{Colors.SUCCESS}All results saved to: {save_dir}{Colors.ENDC}")
    Colors.header("SESSION COMPLETE")

if __name__ == "__main__":
    main()